# Task 1.1 — Core Contribution / Architecture
**Paper:** *Scalable Training of Mixture Models via Coresets* (Feldman, Faulkner, Krause — NIPS 2011)

---

## Step-by-Step Method Description

---

### Step 1: Problem Setup — Define What We Want to Approximate
- **Description:** We have a large dataset D of n points in R^d. We want to fit a Gaussian Mixture Model (GMM) with k components by maximizing the log-likelihood. The key insight is that computing this over all n points is expensive, so we want a small weighted subset C (the *coreset*) that preserves the log-likelihood function up to a (1+ε) factor.
- **Reference:** Section 2, Equation for L(D|θ) and the definition of C_ε (constraint set on eigenvalues).
- **Purpose:** Establishes the mathematical target — we need a coreset C such that φ(D|θ)(1−ε) ≤ φ(C|θ) ≤ φ(D|θ)(1+ε) for *all* θ simultaneously. This is stronger than approximating just the optimal θ.

---

### Step 2: Decompose the Log-Likelihood into a Scale-Invariant Term φ
- **Description:** The raw log-likelihood L(D|θ) has a tricky scaling problem — adding a constant to the data changes the likelihood in a way that breaks multiplicative approximation. The authors decompose L(D|θ) into a normalizer Z(θ) (which doesn't depend on the data) and a term φ(D|θ) that captures all data-dependent parts.
- **Reference:** Section 2, equations splitting L(D|θ) = −n ln Z(θ) + φ(D|θ), where Z(θ) = Σ_i √(w_i / |2πΣ_i|).
- **Purpose:** φ(D|θ) is always non-negative (by Jensen's inequality) and is the correct handle for multiplicative approximation. All coreset guarantees are stated in terms of φ, not L directly.

---

### Step 3: Build an Approximate Clustering B (The "Rough Map")
- **Description:** To avoid the chicken-and-egg problem (needing to know the density to sample, but needing to sample to know the density), the algorithm first builds a rough over-clustering B. It does this iteratively: sample β = O(dk log(1/δ)) points uniformly from the remaining data, then discard the half of the remaining data closest to the already-sampled set. Repeat until the data is small enough. Union everything into B.
- **Reference:** Algorithm 1 (first while-loop), Figure 1(b)(c)(d) showing iterations.
- **Purpose:** B is a coarse approximation with |B| = O(log n) points that roughly covers all clusters. It is not a GMM fit — it's just a geometric scaffold used in the next step.

---

### Step 4: Compute Importance Sampling Probabilities
- **Description:** Each data point x gets an importance weight m(x) that is the sum of two terms: a uniform-within-cluster term (5/|D_b| for the cluster b ∈ B closest to x) and a distance-proportional term (dist(x, B)² / Σ dist(x', B)²). Points far from B get higher probability of being sampled.
- **Reference:** Algorithm 1, the line defining m(x), and Section 3.1 (geometric intuition).
- **Purpose:** This non-uniform sampling reduces variance in the likelihood estimate. Points that are hard to represent (outliers, small clusters far from B) are sampled with higher probability and given lower weights γ(x) to correct for their over-representation.

---

### Step 5: Draw the Coreset C via Non-Uniform Sampling
- **Description:** Sample |C| = O(d·k·|B|²·log(1/δ)/ε²) points from D, where point x is picked with probability proportional to m(x). Each sampled point x' is assigned a corrective weight γ(x') = (Σ m(x)) / (|C| · m(x')), so that the weighted sum over C equals the unweighted sum over D in expectation.
- **Reference:** Algorithm 1 (last three lines), Theorem 3.1.
- **Purpose:** The result is a small weighted dataset C such that for any GMM θ, the weighted log-likelihood on C approximates the full log-likelihood on D within (1±ε). This is the core coreset guarantee.

---

### Step 6: Fit GMM on the Coreset Using Weighted EM
- **Description:** Run a modified EM algorithm on C where the responsibility computation and the M-step updates (means, covariances, weights) are all scaled by the coreset weights γ(x'). The E-step computes soft assignments η_{i,j} for each coreset point to each Gaussian component. The M-step updates parameters using weight-adjusted sufficient statistics.
- **Reference:** Algorithm 2 (Weighted EM), Section 3.3.
- **Purpose:** Because |C| << |D|, this EM runs orders of magnitude faster. The coreset guarantee ensures the solution found on C is nearly as good as the MLE on the full dataset D.

---

### Step 7 (Optional Extension): Streaming / Parallel Coreset Merging
- **Description:** Coresets can be composed: a coreset of a coreset is still a coreset (with slightly larger ε). This allows building coresets in a streaming setting (points arrive one by one) or in a distributed MapReduce setting, using a binary-tree merge-and-compress scheme requiring only O(log n) coresets in memory at any time.
- **Reference:** Section 3.2, Theorems 3.3 and 3.4, Figure 2(b).
- **Purpose:** Extends the method from a batch algorithm to a scalable one that works on data streams and clusters, addressing the original motivation of massive datasets.

---

## Final Summary Sentence

This paper solves the problem of efficiently fitting Gaussian Mixture Models on massive datasets by constructing a small weighted subset (coreset) of size independent of n that provably preserves the log-likelihood up to (1+ε) for *all* possible GMM parameters simultaneously — enabling EM to run on the coreset rather than the full data, achieving speedups of 35×–100× while matching full-data accuracy, which prior uniform subsampling methods fundamentally cannot guarantee.